In [ ]:
#| default_exp cells

# Cells

In [ ]:
#| export
import json
import uuid
import re
import mistune
from fasthtml.common import * 
from llm_sandbox_ui.app_config import PROMPT_SPLIT#, AGENT_CODE_SPLIT


In [ ]:
#| export

up_arrow_ic = NotStr("&#11014")
down_arrow_ic = NotStr("&#11015")
close_ic = NotStr("&#x274C")
swap_ic = NotStr("&#128257")
view_ic =NotStr("&#x1F50D")
clean_ic =NotStr("&#x1F9F9")
minimize_ic = NotStr("&#x25BC");
play_ic = NotStr("&#9654")
stop_ic = NotStr("&#9209")
edit_ic = NotStr("&#x1F4DD")
tools_ic = NotStr("&#x1F6E0")

label_css = "text-xs text-gray-400 px-2 py-1 bg-gray-800"
cell_button_format = 'btn btn-square btn-ghost btn-xs text-xl'


In [ ]:
#| export

def interrupt_button(cell_id):
    """Create an interrupt button that aborts the active stream for a cell.
    
    Args:
        cell_id: Unique cell identifier.
        
    Returns:
        Button component that POSTs to /interrupt/{cell_id}.
    """
    return Button(stop_ic, 
        onClick=f"""fetch('/interrupt/{cell_id}', {{
            method: 'POST', 
            headers: {{'Content-Type': 'application/json'}}, 
            body: JSON.stringify({{notebook: document.querySelector('.notebook-name')?.value || 'untitled'}})
        }})""",
        cls=cell_button_format)
        

In [ ]:
#| export


class BaseCell:
    """Base class for all notebook cell types.
    
    Attributes:
        cell_type (str): Type identifier ('markdown', 'code', 'llm').
        source (str): Cell content/code.
        outputs (list): Cell execution outputs (empty for markdown).
        cell_id (str): Unique cell identifier.
    """
    cell_type = None  
    
    def _make_markdown_init_script(self):
        """ Scripts for managing Markdown display updating """
        return Script(f"""
        (function() {{
            const cell = document.querySelector('[data-cell-id="{self.cell_id}"]');
            if (!cell) return;  // Safety check
            
            const textarea = cell.querySelector('textarea.content-edit');
            const renderdiv = cell.querySelector('.content-render');
            const label = cell.querySelector('.toggle-label');
            const checkbox = cell.querySelector('.toggle-edit');
            
            // Sync textarea to rendered
            textarea.addEventListener('input', () => {{
                renderdiv.innerHTML = marked.parse(textarea.value);
                Prism.highlightAllUnder(renderdiv);
            }});
            
            // Wire label to checkbox
            label.addEventListener('click', () => {{ 
                checkbox.checked = !checkbox.checked; 
                checkbox.dispatchEvent(new Event('change')); 
            }});

            textarea.style.height = 'auto';
            textarea.style.height = Math.min(textarea.scrollHeight, 500) + 'px';
            textarea.addEventListener('input', () => {{{{
                textarea.style.height = 'auto';
                textarea.style.height = Math.min(textarea.scrollHeight, 500) + 'px';
            }}}});

        }})();
        """)

    def __init__(self, source="", outputs="", cell_id=None):
        """Initialize a cell.
        
        Args:
            source (str): Cell content. Defaults to "".
            outputs (list, optional): Execution outputs. Defaults to [].
            cell_id (str, optional): Unique ID. Generates UUID if None.
        """
        self.source = source
        self.cell_id = cell_id or uuid.uuid4().hex[:12]
        self.outputs = outputs 
    
    def build_left_buttons(self):
        """Build cell-specific left button group (play, stop, etc)."""
        pass

    def build_right_buttons(self):
        """Build common right buttons (move up/down/delete)."""
        return Div(
            Div(
                Button(minimize_ic, 
                    onClick=f"toggleMinimize('{self.cell_id}')",
                    cls=cell_button_format),
                Button(up_arrow_ic,
                    cls=cell_button_format,
                    onClick=f"moveUp('{self.cell_id}')"),
                Button(down_arrow_ic,
                    cls=cell_button_format,
                    onClick=f"moveDown('{self.cell_id}')"),
                Button(close_ic,
                    cls=cell_button_format,
                    onClick=f"deleteCell('{self.cell_id}')"),
                cls='btn-group'
            ),
            cls='flex justify-end'
        )

    def build_top_menu(self):
        """Build complete top menu bar with buttons and title."""
        display_name = self.cell_type.replace('_', ' ').title() + ' Cell'
        return Div(
            self.build_left_buttons(),
            Div(display_name, style='font-size: 0.75rem; line-height: 1;'),
            self.build_right_buttons(),
            cls='flex justify-between items-center bg-gray-800 text-white py-0.5 px-2 rounded-t-lg border-b border-gray-700'
        )

    def build_markdown_source_area(self,source,round_b=True):
        """Build textarea and rendered markdown display for a cell.
        
        Args:
            source: Markdown text content.
            round_b: If True, apply rounded bottom corners.
            
        Returns:
            List of [Textarea, Div] components.
        """
        round_cls = 'rounded-b-lg' if round_b else ''

        text_area = Textarea(source,
                    placeholder='Enter Markdown Here...', 
                    rows=1,
                    cls=f'w-full text-gray-100 p-4 bg-[#1e1e1e] {round_cls}'\
                    ' max-h-[500px] overflow-y-auto border-0 content-edit')

        markdown_display = Div(NotStr(mistune.html(source)), 
                            cls='w-full markdown-body bg-gray-900 text-gray-100 p-4'\
                            f' {round_cls} max-h-[500px] overflow-y-auto content-render', 
                            style='list-style-position: inside; min-height: 3.5em;')


        return [text_area, markdown_display]

    def build_monaco_editor(self,cell_id,source_code='', min_height=20, max_height=500):
        """Build Monaco editor initialization script for a code cell.
        
        Args:
            cell_id: Unique cell identifier for the editor container.
            source_code: Initial code content.
            min_height: Minimum editor height in pixels.
            max_height: Maximum editor height in pixels.
            
        Returns:
            Script component that initializes Monaco editor.
        """
        monaco_editor_script = Script(f"""
                                
                require(['vs/editor/editor.main'], function() {{
                    const sourceCode = {json.dumps(source_code)};
                    const container = document.getElementById('monaco-{cell_id}');
                    if (!container) return;
                    
                    const editor = monaco.editor.create(container, {{
                        value: sourceCode,
                        language: 'python',
                        theme: 'vs-dark',
                        automaticLayout: true,
                        scrollBeyondLastLine: false,
                        fontSize: 13, 
                        scrollBeyondLastColumn: 0,
                        model: monaco.editor.createModel(sourceCode, 'python', 
                            monaco.Uri.parse(`inmemory://{cell_id}.py`))
                    }});
                    
                    const lineCount = editor.getModel().getLineCount();
                    const newHeight = Math.min(Math.max(lineCount * 19, {min_height}), {max_height});
                    container.style.height = newHeight + 'px';
                    editor.layout();

                    editor.onDidChangeModelContent(() => {{
                        const lineCount = editor.getModel().getLineCount();
                        const newHeight = Math.min(Math.max(lineCount * 19,{min_height}), {max_height});
                        container.style.height = newHeight + 'px';
                        editor.layout(); 
                    }});
                }});
            
                                """)
        return monaco_editor_script

    def build_code_output(self, tag='',min_height=100, max_height=300, outputs_json=[], round_b = False):
        """Build code output display area with hidden storage.
        
        Args:
            tag: Suffix for CSS class names (e.g., '-code').
            min_height: Minimum output area height in pixels.
            max_height: Maximum output area height in pixels.
            outputs_json: JSON string of Jupyter-style outputs.
            round_b: If True, apply rounded bottom corners.
            
        Returns:
            List of [Pre (display), Div (store)] components.
        """
        if round_b:
            round_button_cls = 'rounded-b-lg' 
        else:
            round_button_cls = ''
            
        output_area_code = Pre(
                          style='font-family: Consolas, Monaco, monospace;'\
                           ' font-size: 13px; background-color: #111827;',
                          cls=f'w-full bg-gray-900 text-gray-100 p-4 {round_button_cls}' \
                          f' min-h-[{min_height}px] max-h-[{max_height}px] overflow-y-auto border-0 output-display{tag}')
                          
        output_store_code = Div(
            outputs_json,  # ← Make sure this has content
            cls='output-store'+tag, 
            style='display:none;'
        )

        return [output_area_code, output_store_code]

    def build_llm_output(self, tag='',min_height=100, max_height=300, outputs_json=""):
        """Build LLM markdown output display area with hidden storage.
        
        Args:
            tag: Suffix for CSS class names (e.g., '-llm').
            min_height: Minimum output area height in pixels.
            max_height: Maximum output area height in pixels.
            outputs_json: Raw markdown string to display.
            
        Returns:
            List of [Div (store), Div (display)] components.
        """
        output_store = Div(
            outputs_json,  # ← Make sure this has content
            cls='output-store'+tag, 
            style='display:none;'
        )

        output_area = Div(
                        cls='w-full markdown-body bg-gray-900 text-gray-100 p-4 rounded-b-lg'\
                        f' min-h-[{min_height}px] max-h-[{max_height}px] overflow-y-auto border-0 output-display{tag}',
                        style='background-color: #111827;',
                        )

        return [output_store, output_area]


    def build_source_area(self):
        """Build editable source code/markdown area."""
        pass

    def build_output_area(self):
        """Build output display area (if applicable)."""
        pass

    def to_ipynb(self):
        """Convert cell to Jupyter notebook dict format."""
        pass

    @classmethod
    def from_ipynb(cls, cell_dict):
        """Create cell from Jupyter notebook dict.
        
        Args:
            cell_dict (dict): Jupyter cell structure.
            
        Returns:
            BaseCell: Instantiated cell subclass.
        """
        pass

    def render(self):
        """Render cell as FastHTML component tree."""
        pass


In [ ]:
#| export

class PromptCell(BaseCell):
    """Cell for LLM prompts with streaming markdown responses.
    
    Stores user prompt in source and LLM response in outputs.
    Serializes to ipynb as markdown with PROMPT_SPLIT separator.
    """
    cell_type = 'prompt'

    @classmethod
    def from_ipynb(cls, cell_dict):
        """ Load From Ipynb Dict"""
        
        source_txt = ''.join(cell_dict['source'])
        if PROMPT_SPLIT in source_txt:
            source, outputs = ''.join(source_txt).split(PROMPT_SPLIT)

        else:
            source = source_txt
            outputs = ''

        return cls(source=source, 
                outputs=outputs,
                cell_id=cell_dict['id'])

    def to_ipynb(self):
        """ Build Ipynb Dict"""

        out_dict = {
            'id':self.cell_id,
            'cell_type': 'markdown',
            'metadata': {'id': self.cell_id, 'prompt_cell': True}
        }
        
        # Combine source + outputs with separator
        if self.outputs:
            source_text = self.source + PROMPT_SPLIT + self.outputs
        else:
            source_text = self.source
        
        out_dict['source'] = source_text.splitlines(keepends=True)
        
        return out_dict

    def build_left_buttons(self):
        """Build cell-specific left button group (play, stop, etc)."""
        
        return Div(
                Div(
                    Button(play_ic,
                            onClick=f"executePromptCell('{self.cell_id}')",
                            cls=cell_button_format),
                    interrupt_button(self.cell_id),
                    Button(clean_ic,
                            onClick=f"clearOutput('{self.cell_id}')",
                            cls=cell_button_format),
                    Label(edit_ic,
                            cls=cell_button_format + ' toggle-label'),
                    Label(
                        Input(type='checkbox',
                            cls="hidden tool-toggle",
                            checked=False),
                            tools_ic,
                            cls=cell_button_format + ' cursor-pointer'),
                    Span(cls="loading loading-spinner loading-sm text-primary cell-spinner hidden"),

                ),
                cls='flex justify-start'
                )

    def build_output_area(self):
        """Build output display area (if applicable)."""
        
        outputs_json = self.outputs if self.outputs else ''
        output_area = self.build_llm_output( tag='',min_height=50, max_height=400, outputs_json=outputs_json)

        llm_out = Div("LLM Output", cls=label_css),

        return [llm_out,*output_area ]

    def render(self):
        """Render cell as FastHTML component tree."""

        toggle_input = Input(type='checkbox', cls="hidden toggle-edit")

        menu_bar = self.build_top_menu()
        source_area = self.build_markdown_source_area(self.source,round_b=False)
        output_area = self.build_output_area()
        
        watch_script = Script(f"""
            setTimeout(() => {{
                watchOutputStore('{self.cell_id}');
            }}, 50);
        """)

        return Div(
            watch_script,
            toggle_input,
            menu_bar,
            *source_area,
            *output_area,
            self._make_markdown_init_script(),
            cls='w-full shadow-xl p-2',
            data_cell_type=self.cell_type,
            data_cell_id=self.cell_id
        )



NameError: name 'BaseCell' is not defined

In [ ]:
#| export

class MarkdownCell(BaseCell):
    """Static markdown documentation cell.
    
    Displays rendered markdown with toggle to edit source.
    No execution or outputs - purely for notes and documentation.
    """
    cell_type = 'markdown'

    @classmethod
    def from_ipynb(cls, cell_dict):
        """ Load From Ipynb Dict"""
                
        source = ''.join(cell_dict['source'])

        return cls(source=source,
                cell_id=cell_dict['id'])

    def to_ipynb(self):
        """ Build Ipynb Dict"""
        
        out_dict = {
            'id':self.cell_id,
            'cell_type': 'markdown',
            'metadata': {'id': self.cell_id},
            'source': self.source.splitlines(keepends=True)
        }
        
        return out_dict

    def build_left_buttons(self):
        """Build cell-specific left button group (play, stop, etc)."""

        toggle_button = Label(edit_ic, cls=cell_button_format + ' toggle-label')
        return Div( toggle_button, cls='flex justify-start')

    def render(self):
        """Render cell as FastHTML component tree."""

        toggle_input = Input(type='checkbox', cls="hidden toggle-edit")

        menu_bar = self.build_top_menu()
        source_area = self.build_markdown_source_area(self.source,round_b=True)


        return Div(
            toggle_input,
            menu_bar,
            *source_area,
            self._make_markdown_init_script(),
            cls='w-full shadow-xl p-2',
            data_cell_type=self.cell_type,  
            data_cell_id=self.cell_id
        )


In [ ]:
#| export

class CodeCell(BaseCell):
    """Executable Python code cell with Monaco editor.
    
    Executes code via kernel and displays Jupyter-style outputs
    (streams, execute_result, display_data, errors).
    """
    cell_type = 'code'
    
    @classmethod
    def from_ipynb(cls, cell_dict):
        """ Load From Ipynb Dict"""
                
        source = ''.join(cell_dict['source'])
        outputs = cell_dict['outputs']

        return cls(source=source,
                outputs = outputs,
                cell_id=cell_dict['id'])

    def to_ipynb(self):
        """ Build Ipynb Dict"""
        
        out_dict = {
            'id':self.cell_id,
            'cell_type': 'code',
            'execution_count': 1,
            'metadata': {'id': self.cell_id},
            'source': self.source.splitlines(keepends=True),
            'outputs': self.outputs
        }
        
        return out_dict

    def build_left_buttons(self):
        """Build cell-specific left button group (play, stop, etc)."""
        
        return Div(
                Div(
                    Button(play_ic,
                            onClick=f"executeCell('{self.cell_id}')",
                            cls=cell_button_format),
                    interrupt_button(self.cell_id),
                    Button(clean_ic,
                            onClick=f"clearOutput('{self.cell_id}')",
                            cls=cell_button_format)
                ),
                cls='flex justify-start'
                )

    def build_source_area(self):
        """Build editable source code/markdown area."""
        
        monaco_editor_script = self.build_monaco_editor(self.cell_id,self.source,min_height=20, max_height=500)

        editor_div = Div(id=f'monaco-{self.cell_id}', 
                        style='height: 20px; width: 100%; overflow: hidden',
                        cls='monaco-editor')
        
        return [editor_div, monaco_editor_script]

    def build_output_area(self):
        """Build output display area (if applicable)."""
        
        outputs_json = json.dumps(self.outputs) if self.outputs else '[]'
        output_area = self.build_code_output(min_height=50, max_height=400, outputs_json=outputs_json , round_b=True)

        code_out = Div("Code Output", cls=label_css),

        return [code_out,*output_area]

    def render(self):
        """Render cell as FastHTML component tree."""

        menu_bar = self.build_top_menu()
        source_area = self.build_source_area()
        output_area = self.build_output_area()

        watch_script = Script(f"""
                setTimeout(() => {{
                    window.cellOutputs['{self.cell_id}'] = window.cellOutputs['{self.cell_id}'] || [];
                    watchOutputStore('{self.cell_id}');
                }}, 50);
                """)
                  
        return Div(
            watch_script,
            menu_bar,
            *source_area,
            *output_area,
            cls='w-full shadow-xl p-2',
            data_cell_type=self.cell_type,
            data_cell_id=self.cell_id
        )


In [ ]:

class AgentCell(BaseCell):
    """Autonomous code generation cell with iterative refinement.
    
    Combines a prompt input, Monaco code editor, and dual output areas
    (unit test results + LLM review). Runs generate→test→improve loops
    until code passes review.
    
    Attributes:
        source_prompt: User's code generation request.
        source_code: Generated/edited Python code.
        outputs_llm: Agent review markdown.
        outputs_code: Unit test execution outputs.
    """
    cell_type = 'agent'


    def __init__(self, source_prompt="", source_code="", outputs="", outputs_llm="",outputs_code="", cell_id=None):
        super().__init__(source="", outputs=outputs, cell_id=cell_id)
        self.source_prompt = source_prompt
        self.source_code = source_code
        self.outputs_llm = outputs_llm
        self.outputs_code = outputs_code

    @classmethod
    def from_ipynb(cls, cell_dict):
        """ Load From Ipynb Dict"""

        source_txt = ''.join(cell_dict['source'])
        code = ""  # ← Add default
        if AGENT_CODE_SPLIT in source_txt:
            prompt, code = source_txt.split(AGENT_CODE_SPLIT)
            re_outputs = re.search(r'```python\n(.*?)\n```', code, re.DOTALL)
            if re_outputs:
                code = re_outputs.group(1)
        else:
            prompt = source_txt

        return cls(source_prompt=prompt, 
                source_code=code,
                cell_id=cell_dict['id'])

    def to_ipynb(self):
        """ Build Ipynb Dict"""

        out_dict = {
            'id':self.cell_id,
            'cell_type': 'markdown',
            'metadata': {'id': self.cell_id, 'agent_cell': True}
        }
        
        # Combine prompt + code with separator
        if self.source_code:
            source_text = self.source_prompt + AGENT_CODE_SPLIT + '```python\n' + self.source_code + '\n```'
        else:
            source_text = self.source_prompt
        
        out_dict['source'] = source_text.splitlines(keepends=True)
        
        return out_dict

    def build_left_buttons(self):
        """Build cell-specific left button group (play, stop, etc)."""
        
        return Div(
                Div(
                    Button(play_ic,
                            onClick=f"executeAgentCell('{self.cell_id}')",
                            cls=cell_button_format),
                    interrupt_button(self.cell_id),
                    Button(clean_ic,
                            onClick=f"clearOutput('{self.cell_id}')",
                            cls=cell_button_format),
                    Label(edit_ic,
                            cls=cell_button_format + ' toggle-label'),
                ),
                cls='flex justify-start'
                )

    def build_source_area(self):
        """Build editable source code/markdown area."""

        markdown_source = self.build_markdown_source_area(self.source_prompt,round_b=False)

        monaco_editor_script = self.build_monaco_editor(self.cell_id,self.source_code,100,300)
       

        editor_div = Div(id=f'monaco-{self.cell_id}', 
                        style='height: 20px; width: 100%; overflow: hidden',
                        cls='monaco-editor')

        agent_in = Div("Prompt", cls=label_css),

        code_in = Div(
            Div("Agent Code", cls="flex-shrink-0"),
            Div("Idle", id=f"status", 
                cls="flex-grow px-3 py-0.5 bg-gray-900 rounded text-center text-xs"),
            Div("Iteration: NA", id=f"iteration",
                cls="flex-shrink-0 px-2 py-0.5 bg-gray-900 rounded text-xs"),
            cls="flex items-center gap-2 text-xs text-gray-400 px-2 py-1 bg-gray-800"
        )


        return [agent_in, *markdown_source,code_in,editor_div, monaco_editor_script]

    def build_output_area(self):
        """Build output display area (if applicable)."""

        ### Code Out
        outputs_json = json.dumps(self.outputs_code) if self.outputs_code else '[]'
        output_area_code = self.build_code_output(tag='-code',min_height=100, max_height=300, outputs_json=outputs_json, round_b=False)

        ### LLM Out
        outputs_json = self.outputs_llm if self.outputs_llm else ''
        output_area_llm = self.build_llm_output( tag='-llm',min_height=100, max_height=300, outputs_json=outputs_json)


        agent_out = Div("Agent Output", cls=label_css),
        code_out = Div("Unit Test Output", cls=label_css),

        return [code_out,*output_area_code,agent_out,*output_area_llm]

    def render(self):
        """Render cell as FastHTML component tree."""

        toggle_input = Input(type='checkbox', cls="hidden toggle-edit")

        menu_bar = self.build_top_menu()
        source_area = self.build_source_area()
        output_area = self.build_output_area()
        
        watch_script = Script(f"""
            setTimeout(() => {{
                watchOutputStore('{self.cell_id}');
            }}, 50);
        """)

        return Div(
            watch_script,
            toggle_input,
            menu_bar,
            *source_area,
            *output_area,
            self._make_markdown_init_script(),
            cls='w-full shadow-xl p-2',
            data_cell_type=self.cell_type,
            data_cell_id=self.cell_id
        )
